# Síntese da revisão sistemática (protocolo v1.4)

Calcula, a partir de `dados/fichamento.xlsx`, as tabelas e figuras do artigo *Existe um método superior para prever a demanda de gás natural? Consenso e comparabilidade em uma revisão sistemática*.
Nenhum número do artigo deve vir de outro lugar.

**Como usar no Colab:** Ambiente de execução › Executar tudo. A primeira célula clona este repositório.
**Como usar localmente:** execute a partir da pasta `notebooks/` do repositório (ou defina a variável `BASE_REVISAO` com o caminho da pasta `dados/`).
Os resultados vão para `resultados/`: `sintese_tabelas.xlsx` (uma aba por tabela) e as figuras em PNG. Nada é gravado em `dados/`.

In [ ]:
# 1. Ambiente e caminhos
import os
BASE = os.environ.get('BASE_REVISAO')
if BASE is None:
    try:
        import google.colab  # noqa: F401  (Colab: clona o repositório público)
        if not os.path.exists('revisao-sistematica-previsao-gn'):
            os.system('git clone -q https://github.com/andersonacs8/revisao-sistematica-previsao-gn.git')
        BASE = 'revisao-sistematica-previsao-gn/dados'
    except ImportError:                  # execução local, a partir de notebooks/
        BASE = os.path.join('..', 'dados')
FICHA = os.path.join(BASE, 'fichamento.xlsx')
SAIDA = os.path.join(BASE, '..', 'resultados')
os.makedirs(SAIDA, exist_ok=True)
print('fichamento:', FICHA, os.path.exists(FICHA))

In [ ]:
# 2. Parâmetros fixados no protocolo v1.4 (seção 8)
# Publicações redundantes: fica a primeira de cada grupo, as demais saem da síntese
REDUNDANTES = {'R49496244': 'R78708291', 'R87996015': 'R78708291',   # Özmen / Başkentgaz
               'R06495962': 'R78776121'}                              # Mulayim 2024 -> 2026
LIMIAR_CONSENSO = 2/3      # família vence em >= 2/3 das comparações neutras -> consenso
FAMILIAS = ['estatístico', 'ML clássico', 'rede neural', 'híbrido', 'fundação']

In [ ]:
# 3. Leitura e limpeza
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 200)
raw = pd.read_excel(FICHA, sheet_name='fichamento', keep_default_na=False, na_values=[''])  # 'n/a' é categoria, não ausente
raw.columns = [str(c).split(' [')[0] for c in raw.columns]
for c in raw.columns[raw.dtypes == object]:
    raw[c] = raw[c].map(lambda v: v.strip() if isinstance(v, str) else v)

def num(v):
    try: return float(v)
    except (TypeError, ValueError): return np.nan
raw['erro_p'] = raw['erro do proposto'].map(num)
raw['erro_c'] = raw['erro do melhor concorrente'].map(num)
# ganho: valor digitado à mão (quando houver) ou calculado a partir dos erros
manual = raw['ganho sobre o melhor concorrente (%)'].map(num)
calc = (raw['erro_c'] - raw['erro_p']) / raw['erro_c'] * 100
raw['ganho'] = manual.where(manual.notna(), calc)
raw['n_conc'] = raw['nº de concorrentes'].map(num)

inc = raw[(raw['entra na síntese?'] == 'I') & (~raw['id'].isin(REDUNDANTES))].copy()
emp = inc[inc['tipo de estudo'] != 'revisão'].copy()          # unidade de análise: estudo empírico/benchmark
A, B = emp[emp['bloco'] == 'A'].copy(), emp[emp['bloco'] == 'B'].copy()
A['grupo'] = np.where(A['tipo de estudo'] == 'benchmark', 'comparação neutra', 'modelo proposto')
print(f"registros: {len(raw)} | incluídos: {len(inc)} | empíricos/benchmark: {len(emp)} (A={len(A)}, B={len(B)})")
print('excluídos por motivo:', raw.loc[raw['entra na síntese?']=='E', 'motivo se não entra'].value_counts().to_dict())

In [ ]:
# 4. Utilitários: tabelas de frequência e exportação
TAB = {}
def freq(df, col, nome=None, ordem=None):
    t = df[col].fillna('(vazio)').value_counts()
    if ordem: t = t.reindex([o for o in ordem if o in t.index] + [i for i in t.index if i not in ordem])
    t = t.to_frame('n'); t['%'] = (100 * t['n'] / t['n'].sum()).round(1)
    TAB[nome or col] = t.reset_index().rename(columns={'index': col}); return t
def resumo(s):
    s = s.dropna()
    return pd.Series({'n': len(s), 'mediana': s.median(), 'Q1': s.quantile(.25), 'Q3': s.quantile(.75)}).round(1)

## Q1–Q3 · Dados, soluções e tendências (bloco A)

In [ ]:
for c in ['unidade prevista', 'tipo de consumidor', 'granularidade', 'horizonte', 'origem dos dados',
          'usa clima?', 'usa calendário?']:
    print(freq(A, c, 'Q1 ' + c), '\n')
for c in ['família do modelo proposto', 'usa decomposição do sinal?', 'usa otimização metaheurística?',
          'um modelo por série ou para várias?']:
    print(freq(A, c, 'Q2 ' + c, FAMILIAS), '\n')

In [ ]:
# Q3: famílias por período de publicação
A['período'] = pd.cut(pd.to_numeric(A['ano']), [2014, 2019, 2022, 2024, 2027], labels=['2015–19', '2020–22', '2023–24', '2025–26'])
q3 = pd.crosstab(A['período'], A['família do modelo proposto']).reindex(columns=[f for f in FAMILIAS if f in A['família do modelo proposto'].unique()])
TAB['Q3 família por período'] = q3.reset_index(); q3

## Q6 · Métricas e P2 · Comparabilidade

In [ ]:
q6 = pd.DataFrame({c: [(A[c] == 'Sim').sum()] for c in ['reporta MAPE?', 'reporta RMSE?', 'reporta MAE?', 'reporta R²?', 'reporta métrica escalada (MASE)?']}, index=['estudos']).T
q6['%'] = (100 * q6['estudos'] / len(A)).round(1); TAB['Q6 métricas reportadas'] = q6.reset_index(); print(q6, '\n')
print(freq(A, 'métrica principal', 'Q6 métrica principal').head(10), '\n')
p2 = pd.Series({
    'compara com ingênuo/sazonal': (A['compara com ingênuo/sazonal?'] == 'Sim').sum(),
    'usa métrica escalada (MASE)': (A['reporta métrica escalada (MASE)?'] == 'Sim').sum(),
    'origem móvel': (A['validação'] == 'origem móvel').sum(),
    'teste de significância': (A['teste de significância?'] == 'Sim').sum(),
    'dados públicos': (A['origem dos dados'] == 'pública').sum(),
    'reporta custo computacional': (A['reporta custo computacional?'] == 'Sim').sum(),
}, name='estudos').to_frame(); p2['%'] = (100 * p2['estudos'] / len(A)).round(1)
TAB['P2 itens de rigor'] = p2.reset_index().rename(columns={'index': 'item'}); p2

## Q7 / P1 · Quem vence, e por quanto

Critério do protocolo (8.1): consenso se uma família vence em ≥ 2/3 das comparações neutras **que a incluem**. O fichamento registra o vencedor, não a lista de famílias testadas em cada estudo; por isso a coluna `% das neutras` usa como denominador **todas** as comparações neutras. É uma aproximação conservadora, a declarar no texto.

In [ ]:
# Família vencedora (= 'proposto') nas comparações neutras x nos estudos com modelo proposto
p1 = pd.crosstab(A['família do modelo proposto'], A['grupo'])
neutras = A[A['grupo'] == 'comparação neutra']
p1['% das neutras'] = (100 * p1.get('comparação neutra', 0) / len(neutras)).round(1)
p1['classificação'] = np.where(p1['% das neutras'] >= 100 * LIMIAR_CONSENSO, 'consenso',
                       np.where(p1['% das neutras'] > 50, 'tendência', 'sem consenso'))
TAB['P1 vencedores por grupo'] = p1.reset_index()
print(f'comparações neutras: {len(neutras)} (resultado indicativo se < 10)'); p1

In [ ]:
# P1 ao longo do tempo: mesmo critério em duas metades (mín. 5 comparações), com IC de Wilson a 95%
import math
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z*z/n; c = (p + z*z/(2*n)) / d; h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return round(100*(c-h)), round(100*(c+h))
N_MIN = 5
metade = np.where(pd.to_numeric(neutras['ano']) <= 2021, '2015–2021', '2022–2026')
blocos = {'2015–2021': neutras[metade == '2015–2021'], '2022–2026': neutras[metade == '2022–2026'], 'Total': neutras}
linhas = {}
for nome, d in blocos.items():
    n = len(d); vc = d['família do modelo proposto'].value_counts()
    col = {}
    for fam in ['estatístico', 'ML clássico', 'rede neural', 'híbrido']:
        k = int(vc.get(fam, 0)); lo, hi = wilson(k, n) if k else (0, 0)
        col[fam] = f'{k} – {round(100*k/n)}% ({lo}–{hi})' if k else '0'
    f = vc.iloc[0] / n; empate = (vc == vc.iloc[0]).sum() > 1
    col['classificação'] = ('n insuficiente' if n < N_MIN else 'consenso' if (f >= LIMIAR_CONSENSO and not empate)
                            else 'tendência' if (f > 0.5 and not empate) else 'sem consenso')
    col['só uma família testada'] = int((d['família do modelo proposto'] == d['classe do melhor concorrente']).sum())
    linhas[f'{nome} (n = {n})'] = col
p1t = pd.DataFrame(linhas); TAB['P1b consenso no tempo'] = p1t.reset_index().rename(columns={'index': 'família'}); p1t

In [ ]:
# Família do melhor concorrente por período (estudos com modelo proposto): o modelo 'a ser batido'
prop_ = A[A['grupo'] == 'modelo proposto']
p1c = pd.crosstab(prop_['período'], prop_['classe do melhor concorrente'], margins=True, margins_name='estudos')
TAB['P1c melhor concorrente no tempo'] = p1c.reset_index(); p1c

In [ ]:
# Ganho sobre o melhor concorrente (dentro de cada estudo)
g = pd.DataFrame({k: resumo(v['ganho']) for k, v in A.groupby('grupo')}).T
gf = pd.DataFrame({k: resumo(v['ganho']) for k, v in A[A['grupo'] == 'modelo proposto'].groupby('família do modelo proposto')}).T
TAB['Q7 ganho por grupo'] = g.reset_index(); TAB['Q7 ganho por família (proposto)'] = gf.reset_index()
print(g, '\n'); print(gf)

## P3 · Indicadores de viés do proposto (padrões compatíveis com viés, não prova)

In [ ]:
prop = A[A['grupo'] == 'modelo proposto']
ind1 = pd.Series({'estudos com ganho calculável': prop['ganho'].notna().sum(),
                  'proposto vence (ganho > 0)': (prop['ganho'] > 0).sum(),
                  'proposto perde ou empata (ganho <= 0)': (prop['ganho'] <= 0).sum()}).to_frame('n')
TAB['P3.1 proposto vence'] = ind1.reset_index().rename(columns={'index': 'indicador'}); print(ind1, '\n')

def estrat(df, col, cortes=None, rotulos=None):
    x = pd.cut(df[col], cortes, labels=rotulos) if cortes is not None else df[col].fillna('(vazio)')
    return pd.DataFrame({k: resumo(v['ganho']) for k, v in df.groupby(x, observed=True)}).T
t4a = estrat(prop, 'n_conc', [-0.5, 0.5, 2.5, 4.5, 100], ['0', '1–2', '3–4', '5+'])
t4b = estrat(prop, 'ajuste dos concorrentes descrito?')
t4c = estrat(prop, 'compara com ingênuo/sazonal?')
t4d = estrat(prop, 'validação')
for nome, t in [('P3.3a ganho x nº concorrentes', t4a), ('P3.3b ganho x ajuste descrito', t4b),
                ('P3.3c ganho x ingênuo', t4c), ('P3.3d ganho x validação', t4d)]:
    TAB[nome] = t.reset_index().rename(columns={'index': 'estrato'}); print(nome); print(t, '\n')
t5 = pd.DataFrame({'como proposto': prop['família do modelo proposto'].value_counts(),
                   'como melhor concorrente': prop['classe do melhor concorrente'].value_counts()}).fillna(0).astype(int)
TAB['P3.4 família proposto x concorrente'] = t5.reset_index().rename(columns={'index': 'família'}); t5

### Ganho ao longo do tempo e por classe do concorrente

In [ ]:
# O ganho declarado não cresce com a migração para modelos complexos (é medido dentro de cada estudo)
gp = pd.DataFrame({k: resumo(v['ganho']) for k, v in prop.groupby(A['período'], observed=True)}).T
TAB['P3.5 ganho por período'] = gp.reset_index().rename(columns={'index': 'período'}); print(gp, '\n')
# Sensibilidade aos cortes: blocos fixos de dois anos e Spearman ano x ganho
from scipy.stats import spearmanr
ano = pd.to_numeric(prop['ano'])
bin2 = pd.cut(ano, [2014, 2016, 2018, 2020, 2022, 2024, 2026], labels=['2015–16','2017–18','2019–20','2021–22','2023–24','2025–26'])
sens = pd.DataFrame({'todos': prop.groupby(bin2, observed=False)['ganho'].median(),
                     'n todos': prop.groupby(bin2, observed=False)['ganho'].count(),
                     'redes': prop[prop['família do modelo proposto']=='rede neural'].groupby(bin2, observed=False)['ganho'].median(),
                     'n redes': prop[prop['família do modelo proposto']=='rede neural'].groupby(bin2, observed=False)['ganho'].count()}).round(1)
ok = prop['ganho'].notna(); nn = ok & (prop['família do modelo proposto']=='rede neural')
r1 = spearmanr(ano[ok], prop.loc[ok,'ganho']); r2 = spearmanr(ano[nn], prop.loc[nn,'ganho'])
print(sens); print(f'Spearman ano x ganho: todos rho={r1.statistic:.2f} p={r1.pvalue:.2f} (n={ok.sum()}); redes rho={r2.statistic:.2f} p={r2.pvalue:.2f} (n={nn.sum()})')
TAB['P3.5b sensibilidade 2 anos'] = sens.reset_index().rename(columns={'index': 'bloco'})
cx = prop[prop['família do modelo proposto'].isin(['rede neural', 'híbrido'])]
gc = pd.DataFrame({k: resumo(v['ganho']) for k, v in cx.groupby('classe do melhor concorrente')}).T
TAB['P3.6 complexo x classe do concorrente'] = gc.reset_index().rename(columns={'index': 'classe do concorrente'}); gc

## P4 · Candidatos ao estado da arte e evidência sobre híbridos

In [ ]:
hib = A[A['família do modelo proposto'] == 'híbrido']
p4h = pd.Series({
    'estudos com híbrido proposto': len(hib),
    'configurações distintas (nome do modelo)': hib['modelo proposto'].nunique(),
    'com decomposição do sinal': (hib['usa decomposição do sinal?'] == 'Sim').sum(),
    'com otimização metaheurística': (hib['usa otimização metaheurística?'] == 'Sim').sum(),
    'comparações neutras vencidas por híbrido': ((A['grupo'] == 'comparação neutra') & (A['família do modelo proposto'] == 'híbrido')).sum(),
    'estudos em que um híbrido é o melhor concorrente': (A['classe do melhor concorrente'] == 'híbrido').sum(),
    'híbridos com origem móvel': (hib['validação'] == 'origem móvel').sum(),
    'híbridos que reportam custo': (hib['reporta custo computacional?'] == 'Sim').sum(),
}, name='n').to_frame()
p4h.loc['ganho mediano do híbrido proposto'] = round(hib.loc[hib['grupo'] == 'modelo proposto', 'ganho'].median(), 1)
TAB['P4 evidência híbridos'] = p4h.reset_index().rename(columns={'index': 'indicador'}); print(p4h, '\n')
# Vencedores das comparações neutras (lista para escolher candidatos, com critério explícito)
p4 = neutras[['id', 'ano', 'família do modelo proposto', 'modelo proposto', 'classe do melhor concorrente', 'ganho']].sort_values('ano')
TAB['P4 vencedores das neutras'] = p4; p4

## P5 · Lacunas e Q8–Q9

In [ ]:
p5 = pd.Series({
    'consumidor industrial individual': ((A['unidade prevista'] == 'consumidor') & (A['tipo de consumidor'] == 'industrial')).sum(),
    'unidade = consumidor (qualquer tipo)': (A['unidade prevista'] == 'consumidor').sum(),
    'modelo global (várias séries)': (A['um modelo por série ou para várias?'] == 'global').sum(),
    'modelo de fundação': (A['família do modelo proposto'] == 'fundação').sum(),
    'compara com ingênuo': (A['compara com ingênuo/sazonal?'] == 'Sim').sum(),
    'MASE': (A['reporta métrica escalada (MASE)?'] == 'Sim').sum(),
    'origem móvel': (A['validação'] == 'origem móvel').sum(),
    'teste de significância': (A['teste de significância?'] == 'Sim').sum(),
    'exigência declarada (Q9)': A['exigência declarada'].notna().sum(),
    'custo computacional (Q8)': (A['reporta custo computacional?'] == 'Sim').sum(),
}, name='estudos').to_frame(); p5['% do bloco A'] = (100 * p5['estudos'] / len(A)).round(1)
TAB['P5 lacunas'] = p5.reset_index().rename(columns={'index': 'aspecto'}); print(p5, '\n')
print(freq(A, 'finalidade da previsão', 'Q9 finalidade'))
TAB['Q9 exigências declaradas'] = inc.loc[inc['exigência declarada'].notna(), ['id', 'bloco', 'exigência declarada']]

## Bloco B · Paradigmas local, global e de fundação (evidência complementar)

In [ ]:
for c in ['um modelo por série ou para várias?', 'família do modelo proposto', 'domínio', 'tipo de estudo']:
    print(freq(B, c, 'B ' + c, FAMILIAS), '\n')
print('Atenção: no bloco B os ganhos não entram na Q7 (muitos valores ausentes e ganhos sobre variantes do mesmo modelo).')

## Figuras e exportação

In [ ]:
import matplotlib.pyplot as plt, matplotlib.ticker as mt
# Estilo único das figuras (paleta validada para daltonismo; identidade também por rótulo direto)
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#8a8984', 'axes.labelcolor': '#52514e', 'xtick.color': '#52514e', 'ytick.color': '#52514e',
    'axes.titleweight': 'bold', 'axes.titlesize': 11, 'axes.titlelocation': 'left', 'figure.facecolor': 'white',
    'axes.grid': True, 'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.axisbelow': True, 'legend.frameon': False})
COR = {'estatístico': '#2a78d6', 'ML clássico': '#eb6834', 'rede neural': '#1baf7a', 'híbrido': '#6250d6'}
TXT, MUT = '#0b0b0b', '#8a8984'

# Figura 1 (opção A): participação de cada família por período, com rótulo direto
fam = [f for f in COR if f in q3.columns]
share = q3[fam].div(q3[fam].sum(axis=1), axis=0) * 100
meio = np.array([2017, 2021, 2023.5, 2025.5])           # ponto médio de cada período (durações diferentes)
fig, ax = plt.subplots(figsize=(7, 4))
for k in fam:
    ax.plot(meio, share[k], color=COR[k], lw=2, marker='o', ms=6, mec='white', mew=1.5)
    ax.annotate(f'{k}  {share[k].iloc[-1]:.0f}%', (meio[-1], share[k].iloc[-1]), xytext=(8, 0),
                textcoords='offset points', va='center', color=TXT, fontsize=9)
ax.set_xticks(meio); ax.set_xticklabels(['2015–2019', '2020–2022', '2023–2024', '2025–2026'])
ax.set_xlim(2016, 2027.8); ax.set_ylim(0, max(50, share.values.max() + 5))
ax.yaxis.set_major_formatter(mt.PercentFormatter(decimals=0)); ax.set_ylabel('% dos estudos do período')
ax.set_title('Família do modelo proposto ao longo do tempo (bloco A)')
ax.text(0, -0.17, 'Pontos no meio de cada período. Estudos por período: ' + ', '.join(map(str, q3[fam].sum(axis=1))),
        transform=ax.transAxes, color=MUT, fontsize=8)
fig.savefig(os.path.join(SAIDA, 'fig_familia_periodo.png'), dpi=300, bbox_inches='tight'); plt.show()

# Figura 2 (opção 1): ganho por estudo, mediana e intervalo interquartil
grupos = {'Comparações neutras': A.loc[A['grupo'] == 'comparação neutra', 'ganho'].dropna(),
          'Estudos com modelo proposto': A.loc[A['grupo'] == 'modelo proposto', 'ganho'].dropna()}
cores = [MUT, '#2a78d6']; rng = np.random.default_rng(2026)
fig, ax = plt.subplots(figsize=(7, 3))
for i, (k, v) in enumerate(grupos.items()):
    ax.scatter(v, i + rng.uniform(-.15, .15, len(v)), s=22, color=cores[i], alpha=.65, edgecolor='white', linewidth=.6)
    q1, md, q3_ = v.quantile([.25, .5, .75])
    ax.plot([q1, q3_], [i + .3, i + .3], color=TXT, lw=2); ax.plot(md, i + .3, 'o', color=TXT, ms=7)
    ax.annotate(f'mediana {md:.0f}%  (n = {len(v)})', (md, i + .3), xytext=(0, 8), textcoords='offset points',
                ha='center', fontsize=9, color=TXT)
ax.set_yticks([0, 1]); ax.set_yticklabels(grupos.keys()); ax.set_ylim(-.5, 1.75)
ax.axvline(0, color=MUT, lw=.8); ax.grid(axis='y', visible=False)
ax.set_xlabel('ganho sobre o melhor concorrente (%)')
ax.set_title('Ganho declarado sobre o melhor concorrente (bloco A)')
fig.savefig(os.path.join(SAIDA, 'fig_ganho_por_grupo.png'), dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
arq = os.path.join(SAIDA, 'sintese_tabelas.xlsx')
with pd.ExcelWriter(arq) as w:
    for nome, t in TAB.items():
        t.to_excel(w, sheet_name=nome[:31].replace('/', '-').replace('?', ''), index=False)
print('gravado:', arq, '|', len(TAB), 'tabelas')